In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta
import pycountry
from numpyro import distributions as dist
import arviz as az
from IPython.display import display, Markdown

from emu_renewal.constants import DATA_PATH, DATE_FORMAT, RUN_DATA_DELAY, RERUNS, FULL_RUN, TIMEOUTS
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.priors import get_standard_priors
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget
from emu_renewal.plotting import plot_input_recovery
from emu_renewal.utils import get_analysis_paths

In [ ]:
mob_source = "fb_singletile_mob"
country = "France"
iso3 = pycountry.countries.lookup(country).alpha_3
name = pycountry.countries.lookup(country).name
continent = get_cont_of_country(iso3)

display(Markdown(f"mobility approach: {mob_source}"))
display(Markdown(f"\n country: {name}"))

In [ ]:
import json

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, countries)

In [ ]:
path = analysis_paths[iso3][mob_source]
idata = az.from_netcdf(path / "idata_filtered.nc")

In [ ]:
post = idata.sample_stats["lp"]
post_array = np.asarray(post)
chain, draw = np.unravel_index(post_array.argmax(), post_array.shape)
idata.posterior.isel(chain=chain, draw=draw)

In [ ]:
# Build the model
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
start_var = "eu"
var_names = [start_var] + alpha_var
seed_times = [] + alpha_seed
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, seed_times, mob_provider, True, False)
thinning = 7
times = model.epoch.number_to_datetime(pd.Series(model.model_times))[::thinning]
# beta_prior = {"beta": dist.Uniform(0.5, 2.5)}
mob_exp_prior = {"mob_exp": dist.Uniform(0.0, 2.0)}

# Use mid-points of priors for calibrated parameters
priors = get_standard_priors(len(var_names), "weekly_admissions", iso3, continent, False)
params = {k: v if isinstance(v, float) else v.mean for k, v in priors.items()}

In [ ]:
# Get synthetic indicators for calibration
for mob_exp in np.linspace(0.5, 1.5, 3):
    identify_params = {"mob_exp": mob_exp}
    proc = np.random.normal(0.0, 0.1, model.x_proc_vals.shape[0])
    results = model.renewal_func(**{"proc": proc} | params | identify_params)
    indicators = ["weekly_deaths", "weekly_cases"]
    outputs = {i: pd.Series(results[i][::thinning], index=times) for i in indicators}
    targets = {ind: SharedDispTarget(targ, weight=20) for ind, targ in outputs.items()}
    calib, mcmc = run_calibration(model, params | identify_params | mob_exp_prior, targets, True, 1000)
    display(plot_input_recovery(priors, calib, az.from_numpyro(mcmc), outputs, identify_params, proc))